In [ ]:
# Imports and Data Split
 
import numpy as np
from pathlib import Path
import joblib
from fingerprints import save_fingerprints

DATA_DIR = "processed_data"

# NSL-KDD (In-Distribution)
X_nsl_train = np.load(f"{DATA_DIR}/X_train.npy")
y_nsl_train = np.load(f"{DATA_DIR}/y_train.npy")

X_nsl_val = np.load(f"{DATA_DIR}/X_val.npy")
y_nsl_val = np.load(f"{DATA_DIR}/y_val.npy")

X_nsl_test = np.load(f"{DATA_DIR}/X_test.npy")
y_nsl_test = np.load(f"{DATA_DIR}/y_test.npy")

# UNSW-NB15 (Out-of-Distribution)
X_unsw_train = np.load(f"{DATA_DIR}/X_ood_train.npy")
y_unsw_train = np.load(f"{DATA_DIR}/y_ood_train.npy")

X_unsw_test = np.load(f"{DATA_DIR}/X_ood_test.npy")
y_unsw_test = np.load(f"{DATA_DIR}/y_ood_test.npy")



In [2]:
print("NSL-KDD")
print("Train:", X_nsl_train.shape, y_nsl_train.shape)
print("Validation:", X_nsl_val.shape, y_nsl_val.shape)
print("Test:", X_nsl_test.shape, y_nsl_test.shape)

print("\nUNSW-NB15")
print("Train:", X_unsw_train.shape, y_unsw_train.shape)
print("Test:", X_unsw_test.shape, y_unsw_test.shape)

print("\nNSL-KDD label distribution")
print("Train:", np.unique(y_nsl_train, return_counts=True))
print("Validation:", np.unique(y_nsl_val, return_counts=True))
print("Test:", np.unique(y_nsl_test, return_counts=True))

print("\nUNSW-NB15 label distribution")
print("Train:", np.unique(y_unsw_train, return_counts=True))
print("Test:", np.unique(y_unsw_test, return_counts=True))

NSL-KDD
Train: (100778, 77) (100778,)
Validation: (25195, 77) (25195,)
Test: (22544, 77) (22544,)

UNSW-NB15
Train: (82332, 77) (82332,)
Test: (175341, 77) (175341,)

NSL-KDD label distribution
Train: (array([0, 1]), array([53874, 46904]))
Validation: (array([0, 1]), array([13469, 11726]))
Test: (array([0, 1]), array([ 9711, 12833]))

UNSW-NB15 label distribution
Train: (array([0, 1]), array([37000, 45332]))
Test: (array([0, 1]), array([ 56000, 119341]))


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

params = {
    "C": [0.01, 0.1, 1, 5, 10, 20, 50],
    "penalty": ["l2"],
    "class_weight": ["balanced"]
}

grid = GridSearchCV(
    LogisticRegression(
        solver="saga",
        max_iter=5000,
        random_state=42
    ),
    param_grid=params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_nsl_train, y_nsl_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)

/opt/homebrew/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' 

Best Parameters: {'C': 20, 'class_weight': 'balanced', 'penalty': 'l2'}
Best CV Accuracy: 0.9344201804934166


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

best_lr_model = grid.best_estimator_

# In-distribution evaluation
nsl_test_predictions = best_lr_model.predict(X_nsl_test)

print("\nTUNED NSL-KDD TEST RESULTS")
print("Accuracy:", accuracy_score(y_nsl_test, nsl_test_predictions))

print("\nConfusion matrix:")
print(confusion_matrix(y_nsl_test, nsl_test_predictions))

print("\nClassification report:")
print(classification_report(
    y_nsl_test,
    nsl_test_predictions,
    target_names=["Normal", "Attack"]
))


===== TUNED NSL-KDD TEST RESULTS =====
Accuracy: 0.7705376153300213

Confusion matrix:
[[9160  551]
 [4622 8211]]

Classification report:
              precision    recall  f1-score   support

      Normal       0.66      0.94      0.78      9711
      Attack       0.94      0.64      0.76     12833

    accuracy                           0.77     22544
   macro avg       0.80      0.79      0.77     22544
weighted avg       0.82      0.77      0.77     22544



In [5]:

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)
train_predictions = best_lr_model.predict(X_nsl_train)

print("Training accuracy:",
      accuracy_score(y_nsl_train, train_predictions))

print("Cross-validation accuracy:",
      grid.best_score_)

print("Test accuracy:",
      accuracy_score(y_nsl_test, nsl_test_predictions))

Training accuracy: 0.9344301335608962
Cross-validation accuracy: 0.9344201804934166
Test accuracy: 0.7705376153300213


In [ ]:
# Out-of-distribution evaluation
unsw_test_predictions = best_lr_model.predict(X_unsw_test)

print("\nUNSW-NB15 OOD TEST RESULTS")
print("Accuracy:", accuracy_score(y_unsw_test, unsw_test_predictions))
print("Confusion matrix:")
print(confusion_matrix(y_unsw_test, unsw_test_predictions))

print("\nClassification report:")
print(classification_report(
    y_unsw_test,
    unsw_test_predictions,
    target_names=["Normal", "Attack"]
))


===== UNSW-NB15 OOD TEST RESULTS =====
Accuracy: 0.3304703406505039
Confusion matrix:
[[22473 33527]
 [83869 35472]]

Classification report:
              precision    recall  f1-score   support

      Normal       0.21      0.40      0.28     56000
      Attack       0.51      0.30      0.38    119341

    accuracy                           0.33    175341
   macro avg       0.36      0.35      0.33    175341
weighted avg       0.42      0.33      0.34    175341



In [ ]:
# Predicted probabilities
nsl_probs = best_lr_model.predict_proba(X_nsl_test)
unsw_probs = best_lr_model.predict_proba(X_unsw_test)

print(nsl_probs.shape)
print(unsw_probs.shape)

(22544, 2)
(175341, 2)


In [8]:
# Confidence
nsl_confidence = np.max(nsl_probs, axis=1)
unsw_confidence = np.max(unsw_probs, axis=1)

print("Average NSL confidence :", np.mean(nsl_confidence))
print("Average UNSW confidence:", np.mean(unsw_confidence))


Average NSL confidence : 0.88654965
Average UNSW confidence: 0.69114804


In [9]:
# Entropy

def entropy(probabilities):
    eps = 1e-12
    probabilities = np.clip(probabilities, eps, 1.0)
    return -np.sum(probabilities * np.log(probabilities), axis=1)

nsl_entropy = entropy(nsl_probs)
unsw_entropy = entropy(unsw_probs)

print("Average NSL entropy :", np.mean(nsl_entropy))
print("Average UNSW entropy:", np.mean(unsw_entropy))

Average NSL entropy : 0.28320217
Average UNSW entropy: 0.5492662


In [10]:
sorted_nsl = np.sort(nsl_probs, axis=1)
sorted_unsw = np.sort(unsw_probs, axis=1)

nsl_margin = sorted_nsl[:, -1] - sorted_nsl[:, -2]
unsw_margin = sorted_unsw[:, -1] - sorted_unsw[:, -2]

print("Average NSL margin :", np.mean(nsl_margin))
print("Average UNSW margin:", np.mean(unsw_margin))

Average NSL margin : 0.77309924
Average UNSW margin: 0.3822961


In [11]:
def expected_calibration_error(y_true, probs, n_bins=10):

    confidence = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracy = (predictions == y_true)

    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        mask = (confidence > bins[i]) & (confidence <= bins[i+1])

        if np.sum(mask) > 0:
            bin_accuracy = np.mean(accuracy[mask])
            bin_confidence = np.mean(confidence[mask])

            ece += (np.sum(mask) / len(y_true)) * abs(bin_accuracy - bin_confidence)

    return ece

In [12]:
# Calc Expected Calibration Error (ECE)
ece_nsl = expected_calibration_error(y_nsl_test, nsl_probs)
ece_unsw = expected_calibration_error(y_unsw_test, unsw_probs)

print("NSL ECE :", ece_nsl)
print("UNSW ECE:", ece_unsw)

NSL ECE : 0.11601200423251662
UNSW ECE: 0.3606776872815837


In [ ]:
OUTPUT_DIR = Path("model_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# Create output directory
model_dir = Path("model_outputs/logistic_regression_model")
model_dir.mkdir(parents=True, exist_ok=True)

# Save the best Logistic Regression model
joblib.dump(
    best_lr_model,
    model_dir / "best_logistic_regression_model.joblib"
)

print("Saved model:", model_dir / "best_logistic_regression_model.joblib")

Saved model: model_outputs/logistic_regression_model/best_logistic_regression_model.joblib


In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV


svm = SVC(
    probability=True,   # predict_proba() works later for fingerprints
    random_state=42
)

param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}

grid = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X_nsl_train, y_nsl_train)

best_svm_model = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best cross-validation F1:", grid.best_score_)

model_dir = Path("model_outputs/svm_model")
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    best_svm_model,
    model_dir / "best_svm_model.joblib"
)

print("Saved model:", model_dir / "best_svm_model.joblib")

Best parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Best cross-validation F1: 0.9636355582652779
Saved model: model_outputs/svm_model/best_svm_model.joblib


In [ ]:
best_svm_model = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best cross-validation F1:", grid.best_score_)

model_dir = Path("model_outputs/svm_model")
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    best_svm_model,
    model_dir / "best_svm_model.joblib"
)

print("Saved model:", model_dir / "best_svm_model.joblib")

Best parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Best cross-validation F1: 0.9574741709315875
Saved model: model_outputs/svm_model/best_svm_model.joblib


In [43]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score
)

nsl_svm_predictions = best_svm_model.predict(X_nsl_test)

print("\nSVM — NSL-KDD Test")
print("Accuracy:", accuracy_score(y_nsl_test, nsl_svm_predictions))
print("F1 score:", f1_score(y_nsl_test, nsl_svm_predictions))
print("\nConfusion Matrix:")
print(confusion_matrix(y_nsl_test, nsl_svm_predictions))
print("\nClassification Report:")
print(classification_report(y_nsl_test, nsl_svm_predictions))


SVM — NSL-KDD Test
Accuracy: 0.8346344925479063
F1 score: 0.8353356890459364

Confusion Matrix:
[[9360  351]
 [3377 9456]]

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.96      0.83      9711
           1       0.96      0.74      0.84     12833

    accuracy                           0.83     22544
   macro avg       0.85      0.85      0.83     22544
weighted avg       0.87      0.83      0.83     22544



In [44]:
unsw_svm_predictions = best_svm_model.predict(X_unsw_test)

print("\nSVM — UNSW-NB15 Test")
print("Accuracy:", accuracy_score(y_unsw_test, unsw_svm_predictions))
print("F1 score:", f1_score(y_unsw_test, unsw_svm_predictions))
print("\nConfusion Matrix:")
print(confusion_matrix(y_unsw_test, unsw_svm_predictions))
print("\nClassification Report:")
print(classification_report(y_unsw_test, unsw_svm_predictions))


SVM — UNSW-NB15 Test
Accuracy: 0.2666632447630617
F1 score: 0.2527661552766155

Confusion Matrix:
[[25009 30991]
 [97593 21748]]

Classification Report:
              precision    recall  f1-score   support

           0       0.20      0.45      0.28     56000
           1       0.41      0.18      0.25    119341

    accuracy                           0.27    175341
   macro avg       0.31      0.31      0.27    175341
weighted avg       0.35      0.27      0.26    175341



In [ ]:
#Import and tune the MLP model

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV

mlp = MLPClassifier(
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=42
)

mlp_param_grid = {
    "hidden_layer_sizes": [
        (64,),
        (128,),
        (64, 32)
    ],
    "activation": ["relu"],
    "alpha": [0.0001, 0.001],
    "learning_rate_init": [0.001]
}

mlp_grid = GridSearchCV(
    estimator=mlp,
    param_grid=mlp_param_grid,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    verbose=2
)

mlp_grid.fit(X_nsl_train, y_nsl_train)

best_mlp_model = mlp_grid.best_estimator_

print("Best parameters:", mlp_grid.best_params_)
print("Best cross-validation F1:", mlp_grid.best_score_)

Fitting 3 folds for each of 6 candidates, totalling 18 fits
[CV] END activation=relu, alpha=0.0001, hidden_layer_sizes=(64, 32), learning_rate_init=0.001; total time=   5.8s
[CV] END activation=relu, alpha=0.0001, hidden_layer_sizes=(64, 32), learning_rate_init=0.001; total time=   7.1s
[CV] END activation=relu, alpha=0.0001, hidden_layer_sizes=(128,), learning_rate_init=0.001; total time=  11.1s
[CV] END activation=relu, alpha=0.0001, hidden_layer_sizes=(64,), learning_rate_init=0.001; total time=  14.3s
[CV] END activation=relu, alpha=0.0001, hidden_layer_sizes=(64,), learning_rate_init=0.001; total time=  15.3s
[CV] END activation=relu, alpha=0.0001, hidden_layer_sizes=(128,), learning_rate_init=0.001; total time=  15.9s
[CV] END activation=relu, alpha=0.0001, hidden_layer_sizes=(64,), learning_rate_init=0.001; total time=  16.6s
[CV] END activation=relu, alpha=0.0001, hidden_layer_sizes=(64, 32), learning_rate_init=0.001; total time=  12.9s
[CV] END activation=relu, alpha=0.0001, h

In [ ]:
mlp_model_dir = Path("model_outputs/mlp_model")
mlp_model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    best_mlp_model,
    mlp_model_dir / "best_mlp_model.joblib"
)

print(
    "Saved model:",
    mlp_model_dir / "best_mlp_model.joblib"
)

Saved model: model_outputs/mlp_model/best_mlp_model.joblib


In [23]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score
)

nsl_mlp_predictions = best_mlp_model.predict(X_nsl_test)

print("\nMLP — NSL-KDD Test")
print("Accuracy:", accuracy_score(y_nsl_test, nsl_mlp_predictions))
print("F1 score:", f1_score(y_nsl_test, nsl_mlp_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_nsl_test, nsl_mlp_predictions))

print("\nClassification Report:")
print(classification_report(y_nsl_test, nsl_mlp_predictions))


MLP — NSL-KDD Test
Accuracy: 0.8347232079489
F1 score: 0.8356997971602435

Confusion Matrix:
[[9342  369]
 [3357 9476]]

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.96      0.83      9711
           1       0.96      0.74      0.84     12833

    accuracy                           0.83     22544
   macro avg       0.85      0.85      0.83     22544
weighted avg       0.86      0.83      0.83     22544



In [24]:
unsw_mlp_predictions = best_mlp_model.predict(X_unsw_test)

print("\nMLP — UNSW-NB15 Test")
print("Accuracy:", accuracy_score(y_unsw_test, unsw_mlp_predictions))
print("F1 score:", f1_score(y_unsw_test, unsw_mlp_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_unsw_test, unsw_mlp_predictions))

print("\nClassification Report:")
print(classification_report(y_unsw_test, unsw_mlp_predictions))


MLP — UNSW-NB15 Test
Accuracy: 0.4097957693865097
F1 score: 0.5365167658690171

Confusion Matrix:
[[11957 44043]
 [59444 59897]]

Classification Report:
              precision    recall  f1-score   support

           0       0.17      0.21      0.19     56000
           1       0.58      0.50      0.54    119341

    accuracy                           0.41    175341
   macro avg       0.37      0.36      0.36    175341
weighted avg       0.45      0.41      0.43    175341



# Balanced split + fingerprint generation for Layer 2



- **NSL `X_test`** is split once (70/30) into a detector-training slice and a genuinely held-out
  **blind-test** slice.
- **UNSW** uses two already-separate, never-mixed pools: `X_ood_test` for detector training
  ("Portion 1") and `X_ood_train` for the blind test ("Portion 2") — `X_ood_train` has never been
  touched by anything up to this point.
- Both UNSW pools are **subsampled** to roughly 2x the NSL side, so Layer 2 doesn't train (or get
  blind-tested) on a wildly imbalanced 1:11 ratio.
- **`X_val` / `y_val` is intentionally left untouched here** — it's reserved for temperature scaling,
  not for Layer 2 training or evaluation.

Output: four fingerprint CSVs per model —
`nsl_detector_train`, `unsw_portion1` (→ Layer 2 training),
`nsl_blind_test`, `unsw_portion2` (→ final blind test).

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

#  Split NSL X_test/y_test once (70% detector-train / 30% blind-test), stratified
X_test_train, X_test_blind, y_test_train, y_test_blind = train_test_split(
    X_nsl_test, y_nsl_test,
    test_size=0.30,
    stratify=y_nsl_test,
    random_state=RANDOM_STATE,
)

print("NSL detector-train:", X_test_train.shape, "| label counts:", np.unique(y_test_train, return_counts=True))
print("NSL blind-test:    ", X_test_blind.shape, "| label counts:", np.unique(y_test_blind, return_counts=True))


NSL detector-train: (15780, 77) | label counts: (array([0, 1]), array([6797, 8983]))
NSL blind-test:     (6764, 77) | label counts: (array([0, 1]), array([2914, 3850]))


In [ ]:
#  Subsample UNSW pools to roughly 2x the NSL side (avoids extreme class imbalance in Layer 2)

n_train_target = len(X_test_train)   # NSL detector-train size -> sets UNSW Portion 1 sample size
n_blind_target = len(X_test_blind)   # NSL blind-test size     -> sets UNSW Portion 2 sample size

rng = np.random.default_rng(RANDOM_STATE)

def subsample(X, y, n):
    n = min(n, len(X))
    idx = rng.choice(len(X), size=n, replace=False)
    return X[idx], y[idx]

# UNSW Portion 1 (detector training) 
X_unsw_portion1, y_unsw_portion1 = subsample(X_unsw_test, y_unsw_test, n_train_target * 2)

# UNSW Portion 2 (blind test) 
X_unsw_portion2, y_unsw_portion2 = subsample(X_unsw_train, y_unsw_train, n_blind_target * 2)

print("UNSW Portion 1 (train):", X_unsw_portion1.shape, "| label counts:", np.unique(y_unsw_portion1, return_counts=True))
print("UNSW Portion 2 (blind):", X_unsw_portion2.shape, "| label counts:", np.unique(y_unsw_portion2, return_counts=True))


UNSW Portion 1 (train): (31560, 77) | label counts: (array([0, 1]), array([10082, 21478]))
UNSW Portion 2 (blind): (13528, 77) | label counts: (array([0, 1]), array([5961, 7567]))


In [ ]:
# 3. Generate fingerprints for all four pieces, per base model
# No retraining happens here

base_models = {
    "logistic_regression": best_lr_model,
    "svm": best_svm_model,
    "mlp": best_mlp_model,
}

for model_name, model in base_models.items():
   
    save_fingerprints(model, X_test_train, y_test_train, model_name, "nsl_detector_train")
    save_fingerprints(model, X_unsw_portion1, y_unsw_portion1, model_name, "unsw_portion1")

   
    save_fingerprints(model, X_test_blind, y_test_blind, model_name, "nsl_blind_test")
    save_fingerprints(model, X_unsw_portion2, y_unsw_portion2, model_name, "unsw_portion2")



logistic_regression — nsl_detector_train
Average confidence: 0.88607347
Average entropy: 0.28364232
Average margin: 0.77214694
ECE: 0.11655509726856148

Saved CSV: model_outputs/logistic_regression_fingerprints/nsl_detector_train_fingerprints.csv
Saved NumPy: model_outputs/logistic_regression_fingerprints/nsl_detector_train_fingerprints.npy

logistic_regression — unsw_portion1
Average confidence: 0.6908189
Average entropy: 0.54962933
Average margin: 0.38163775
ECE: 0.36236511961248896

Saved CSV: model_outputs/logistic_regression_fingerprints/unsw_portion1_fingerprints.csv
Saved NumPy: model_outputs/logistic_regression_fingerprints/unsw_portion1_fingerprints.npy

logistic_regression — nsl_blind_test
Average confidence: 0.8876605
Average entropy: 0.2821752
Average margin: 0.775321
ECE: 0.11474507610907855

Saved CSV: model_outputs/logistic_regression_fingerprints/nsl_blind_test_fingerprints.csv
Saved NumPy: model_outputs/logistic_regression_fingerprints/nsl_blind_test_fingerprints.npy


**Note:** `X_val` / `y_val` were loaded at the top of this notebook but deliberately not used anywhere
above -- they're reserved for temperature scaling (picking the best `T` by minimizing ECE), which is a
separate step from Layer 2 detector training/blind-testing done here.